In [1]:
from dvc.fs import download

from src.susse.api_clients.cams.cams_client import CAMSClient
%load_ext autoreload
%autoreload 2
# %matplotlib widget
import sys
from pathlib import Path

src_path = (Path.cwd() / "../src/susse").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

/home/jan/.pyenv/versions/.venv_irradiation/lib/python3.11/site-packages/pvlib/spectrum/mismatch.py:10: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.integrate import trapezoid


# Preparation
In this notebook, we show how we can query all the necessary information for the model training for a specific location for a specific time period.

In [4]:
from geopy import location as Glocation
from geopy.geocoders import Nominatim

from datetime import datetime, timedelta

start_date = datetime(2022, 1, 1)
end_date = datetime(2022, 2, 1)

geolocator = Nominatim(user_agent="my_geocoder_app")
analysis_location = geolocator.geocode("Kampala, UG")

# CAMS
We can query irradiation estimates from CAMS using the CAMSClient class. Note, that in order to be able to query data from CAMS, the user needs to have an email address and password. The data is available through the browser: https://ads.atmosphere.copernicus.eu/datasets/cams-solar-radiation-timeseries?tab=download

This is also, where the user can create a password and username

In [ ]:
from src.susse.api_clients import CAMSClient
from datetime import datetime

cams_client = CAMSClient()
out_data, metadata = cams_client.fetch_data(analysis_location.latitude, analysis_location.longitude, start_date, end_date, "1d")
out_data.head()

# NASA MERRA 2

The NASA Merra 2 (Modern-Era Retrospective analysis for Research and Applications, Version 2) project, provides retrospective athmospheriacal data. In order to create a training dataset, we query several of these variables. We can use the MerraDownloadManager class

In [ ]:
from src.susse.api_clients import MerraDataFetcher, MerraProducts

merra_fetcher = MerraDataFetcher(".")
merra_data = merra_fetcher.fetch_product_result(MerraProducts.EASTWARD_WIND.value, analysis_location, start_date, end_date)

merra_data.to_df().head()

# NASA Modis
The same can be done with the NASA MODIS (Moderate Resolution Imaging Spectroradiometer) data (https://modis.gsfc.nasa.gov/)

In [ ]:
from src.susse.api_clients import ModisDataFetcher, ModisProductEnum

modis_data_fetcher = ModisDataFetcher()
modis_result = modis_data_fetcher.fetch_product_result(ModisProductEnum.LAND_SURFACE_TEMPERATURE_DAILY, analysis_location, start_date, end_date)
modis_result.to_df().head()